# Lifted Wheel Dynamics

In [ ]:
import sympy as sm
import sympy.physics.mechanics as me

me.init_vprinting(use_latex='mathjax')

## Symbols

- $t$ time
- $m$ Wheel mass (kg)
- $r_w$ Wheel radius (m) ($r_{w,\text{inner}}$ Inner low density region)
- $h_w$ Wheel height (m)
---
- $\theta$ wheel angle
- $\omega$ Wheel angular speed
- $T_{m,w}$ Motor Torque (Wheelside of gearbox) (Nm)

In [ ]:
subs_dict = {}

t = me.dynamicsymbols._t
m, r_wheel, h_wheel, r_w_inner = sm.symbols('m r_w h_w r_w_inner', positive=True, real=True)
theta, omega = me.dynamicsymbols('theta omega', real=True)
T_motor_w = me.dynamicsymbols('T_{m\\,w}', real=True)

In [ ]:
# Theta derivatives

theta_dot = theta.diff()
subs_dict[theta_dot] = omega

display(sm.Eq(theta_dot, subs_dict[theta_dot]))

# Frame definition

$N$ Inertail frame

$D$ is the wheel frame, rotating with the wheel (left wheel, pos $\theta$ drives forward)

In [ ]:
N = me.ReferenceFrame('N')
D = me.ReferenceFrame('D')

D.orient_axis(N, N.y, theta)

In [ ]:
I_wheel = me.inertia(
    frame=D,
    iyy=m*(r_wheel**2 + r_w_inner**2)/2,
    ixx=m*((r_wheel**2 + r_w_inner**2)*3+(h_wheel**2))/12,
    izz=m*((r_wheel**2 + r_w_inner**2)*3+(h_wheel**2))/12,
)


In [ ]:
# sm.Eq(I_wheel.express(N) * D.ang_acc_in(N), t* D.y)
N_w_D = D.ang_vel_in(N)


# https://moorepants.github.io/learn-multibody-dynamics/generalized-forces.html 171
# Calculating from Bo Body center of anass so no parallel axis theorem??? CHECK AGAIN
# Ignoring gravity related axle bending load effects
T_total = (I_wheel.dot(D.ang_acc_in(N)) - T_motor_w*N.y).express(D)
T_total

In [ ]:
# me.outer(D.y, D.y)
omega_dot_sol = sm.solve(T_total.dot(D.y).subs(subs_dict), omega.diff())

display(sm.Eq(omega.diff(), omega_dot_sol[0]))

# Motor equations
Source: wb2232 lecture 2a - Tuesday 15 feb 2022
Wikipedia

----

$$T_{m,\text{all}} = k_t I = \frac{k_t}{R}(U_\text{in} - k_t\omega_{m,\text{internal}})$$

- $k_t$ Motor constante
- $I$ Motor current ($A$)
- $R$ Motor Resitantce ($\Omega$)
- $\omega_{m,\text{internal}}$ Interan motor speed

$$T_{m,\text{use}} = T_{m,\text{all}} - T_0 = \frac{k_t}{R}(U_\text{in} - k_t\omega_{m,\text{internal}}) - T_0$$

- $T_0$ motorloss
----


# Gearbox

$$P_\text{mech} = T_{m,w}\omega = T_{m,\text{use}}\omega_{m,\text{internal}}$$

$$\frac{T_{m,w}}{T_{m,\text{use}}} = \frac{\omega_{m,\text{internal}}}{\omega} = N$$

$$T_{m,w} = N T_{m,\text{use}}$$
$$\omega_{m,\text{internal}} = N \omega$$

In [ ]:
kt, R, T0, N = sm.symbols('k_t R T_0 N', real=True, positive=True)
Uin = sm.symbols('U_{in}', real=True)

#### **THIS IS INCORRECT, SINCE $T_0$ IS A MAXIMUM AND NOT CONSTANTLY APPLIED, _HOWEVER IT MODELS IT ACCURATE ENOUGH FOR NOW_**

In [ ]:
# T_motor_expr = 
omega_internal = N * omega

T_motor_all = kt/R * (Uin - kt*omega_internal)
# T_motor_w_expr = (N * (T_motor_all - sm.Piecewise((T0, T_motor_all >= T0), (T_motor_all, T0>T_motor_all))))# .simplify()
# T_motor_w_expr = (N * (T_motor_all - sm.Min(T0, T_motor_all))).simplify()
T_motor_w_expr = (N * (T_motor_all - T0)).simplify()
subs_dict[T_motor_w] = T_motor_w_expr

sm.Eq(T_motor_w, subs_dict[T_motor_w])

In [ ]:
omega_dot_eq = sm.Eq(omega.diff(), omega_dot_sol[0].subs(subs_dict))
display(omega_dot_eq)

| Symbol | Value | Source         |
|:------:|:-----:|:--------------:|
|$U_{in}$| 12 V  | Specs (nominal)|
| $N$    | 30   | AliExpress     |
| 


In [ ]:
sm.classify_ode(omega_dot_eq, omega)

#### **THIS IS INCORRECT, SINCE $T_0$ IS A MAXIMUM AND NOT CONSTANTLY APPLIED, _HOWEVER IT MODELS IT ACCURATE ENOUGH FOR NOW_**

In [ ]:
omega_sol_final = sm.dsolve(omega_dot_eq, omega, ics={omega.subs(t, 0): 0})
display(omega_sol_final)

## More motor currents
### No Load
- $\omega_{w,\text{no load}} = 200 \text{RPM}$
- $I_\text{no load} = 70 mA$

(Use $T_{m,\text{all}} = k_t I$)
$$T_{m,\text{use}} = 0 = \frac{k_t}{R}({U_\text{in}-k_t\omega_{m,\text{internal},\text{no load}}}) - T_0 = k_t I_\text{no load} - T_0$$
$$T_0 = k_t I_\text{no load}$$

$$\frac{k_t}{R}({U_\text{in}-k_t\omega_{m,\text{internal},\text{no load}}}) = k_t I_\text{no load}$$
$$U_\text{in}-k_t\omega_{m,\text{internal},\text{no load}} = R I_\text{no load}$$
<!-- $$T_{m,\text{all}} = T_0 =  \frac{k_t}{R}({\underline{U_\text{in}}-k_t\underline{\omega_{m,\text{internal}}}}) - T_0$$ -->
<!-- $$\underline{T_0} = \frac{k_t}{R}({\underline{U_\text{in}}-k_t\underline{\omega_{m,\text{internal}}}})$$ -->


### Stall
- $T_{m,w,\text{stall}} = 2.5 \text{kg cm}$
- $I_\text{stall} = 1.2 A$
$$T_{m,\text{use},\text{stall}} = \frac{k_t}{R}(U_\text{in}) - T_0 = k_t I_\text{stall} - T_0 $$

### At load (specsheet)
- $T_{m,w,\text{load}} = 1.7 \text{kg cm}$
- $\omega_{w,\text{load}} = 166 \text{RPM}$
- $I_\text{load} = 0.45 A$

$$T_{m,\text{use},\text{load}} = \frac{k_t}{R}(U_\text{in} - k_t\omega_{m,\text{internal},\text{load}}) - T_0 = k_t I_\text{load} - T_0$$


In [ ]:
motor_subs_dict = {}

T_motor_wheel_stall, T_motor_wheel_load = sm.symbols(r'T_{m\,w\,\text{stall}} T_{m\,w\,\text{load}}', real=True)
I_no_load, I_stall, I_load = sm.symbols(r'I_\text{no\ load} I_\text{stall} I_\text{load}', real=True)
omega_w_no_load, omega_w_load = sm.symbols(r'\omega_{w\,\text{no\ load}} \omega_{w\,\text{load}}', real=True)

Uin_max = sm.symbols(r'U_{in\,\text{max}}', real=True)

In [ ]:
# No Load
equ_no_load1 = sm.Eq(kt/R * (Uin_max - kt * N * omega_w_no_load) - T0, kt*I_no_load - T0)
display(equ_no_load1)

equ_no_load2 = sm.Eq(0, kt*I_no_load - T0)  # Used
display(equ_no_load2)

In [ ]:
motor_subs_dict[R] = sm.solve(equ_no_load1, R)[0]
display(sm.Eq(R, motor_subs_dict[R]))

In [ ]:
# Stall
equ_stall1 = sm.Eq(T_motor_wheel_stall, N*(kt/R*(Uin_max - kt*0) - T0)).subs(motor_subs_dict)
display(equ_stall1)

equ_stall2 = sm.Eq(T_motor_wheel_stall, N *(kt*I_stall - T0)) # Used
display(equ_stall2.subs(motor_subs_dict))

In [ ]:
motor_subs_dict[kt] = sm.solve(equ_stall2.subs(motor_subs_dict), kt)[0]
display(sm.Eq(kt, motor_subs_dict[kt]))

In [ ]:
# At load
equ_at_load1 = sm.Eq(T_motor_wheel_load, N*(kt/R*(Uin_max - kt*N*omega_w_load)-T0))
display(equ_at_load1)

In [ ]:
motor_subs_dict[T0] = sm.solve(equ_at_load1.subs(motor_subs_dict), T0)[0]  # There are 2 this should be the correct one
display(sm.Eq(T0, motor_subs_dict[T0]))

In [ ]:
MOTOR_LISTING_ALIEXPRESS: bool = True
kgcm_to_Nm_factor = sm.Rational('0.0980665')

if MOTOR_LISTING_ALIEXPRESS:
    MOTOR_VALUE_DICT = {
        N: sm.Rational('30'),
        I_no_load:  sm.Rational('70e-3'),
        I_stall: sm.Rational('1.2'),
        I_load: sm.Rational('0.45'),
        omega_w_load: sm.pi * 166 * 2 / 60,
        omega_w_no_load: sm.pi * 200 * 2 / 60,
        T_motor_wheel_stall: sm.Rational('2.5') * kgcm_to_Nm_factor,
        T_motor_wheel_load: sm.Rational('1.7') * kgcm_to_Nm_factor,
        Uin_max: sm.Rational(12),
    }
else:  # https://www.alibaba.com/product-detail/JGB37-520-kit-metal-electric-toy_1600612543054.html
    # https://www.aliexpress.us/item/2255801080362575.html?gatewayAdapt=glo2usa4itemAdapt
    MOTOR_VALUE_DICT = {
        N: sm.Rational('30'),
        omega_w_no_load: sm.pi * 200 * 2 / 60,
        I_no_load: sm.Rational('0.04'),
        omega_w_load: sm.pi * 154 * 2 /60,
        I_load: sm.Rational('0.3'),
        T_motor_wheel_load: sm.Rational('0.8') * kgcm_to_Nm_factor,
        T_motor_wheel_stall: sm.Rational('3.5') * kgcm_to_Nm_factor,
        Uin_max: sm.Rational(12),
        I_stall: sm.Rational('1.4'),
    }

In [ ]:
VALUE_DICT = {
    Uin: sm.Rational('12'),
    r_wheel: sm.Rational('97e-3')/2,
    r_w_inner: sm.Rational('42.5e-3'),  # Simulating most mass on outside
    # m: sm.Rational('573.6e-3')/4,  # https://yfrobot.com/products/plastic-mecanum-wheels-97mm?variant=43078442188962
    m: sm.Rational('162e-3'),  # https://www.amazon.nl/Mecanum-Wheel-Omni-directionele-Smart-Robot/dp/B0CS5QSHPN?th=1
}
VALUE_DICT.update(MOTOR_VALUE_DICT)


In [ ]:
# omega_dot_eq.subs(motor_subs_dict).subs(VALUE_DICT).expand().simplify()
# omega_sol_final = sm.dsolve(omega_dot_eq.subs(motor_subs_dict).subs(VALUE_DICT), omega)

In [ ]:
display(omega_sol_final.subs(subs_dict).subs(motor_subs_dict).subs(motor_subs_dict).subs(VALUE_DICT).evalf().simplify())

In [ ]:
# C1 = list(set(
#         filter(
#             lambda s: isinstance(s, sm.Symbol),
#             omega_sol_final.subs(motor_subs_dict).subs(VALUE_DICT).atoms(),
#         ),
#     ).difference({t}))[0]

In [ ]:
solution = sm.lambdify((t), omega_sol_final.subs(subs_dict).subs(motor_subs_dict).subs(motor_subs_dict).subs(VALUE_DICT).rhs)

In [ ]:
# Still too fast
# Probably missing the motor rotor/axle/stator inertia.
solution(0.05)

In [ ]:
display(sm.Eq(T0, motor_subs_dict[T0].subs(motor_subs_dict)).subs(VALUE_DICT).evalf())

In [ ]:
display(T_motor_w_expr)#.subs(motor_subs_dict).subs(subs_dict).simplify())
display(
    T_motor_w_expr
        .subs(motor_subs_dict)
        .subs(motor_subs_dict)
        .subs(subs_dict)
        .subs({
            omega: 0,
            Uin: Uin_max,
        })
        .simplify().subs(VALUE_DICT).evalf())

display(VALUE_DICT[T_motor_wheel_stall].evalf())

# Check with Power

$$P_{el} = U_{in} I_{in} = U_{in}\frac{T_{m,\text{all}}}{k_t} = U_{in}\frac{T_{m,\text{all}}}{k_t}$$

$$ P_{m,me} = T_{m,use} \omega_{m,\text{internal}}$$

$$\mu_m
    = \frac{P_{m,me}}{P_{el}}
    = \frac{T_{m,use} \omega_m}{U_{in}\frac{T_{m,\text{all}}}{k_t}}
    = \frac{k_t T_{m,use} \omega_m}{U_{in}{T_{m,\text{all}}}}
    = \frac{k_t (T_{m,\text{all}} - T_0) \omega_m}{U_{in}{T_{m,\text{all}}}}
    =   \frac{k_t T_{m,\text{all}} \omega_m}{U_{in}{T_{m,\text{all}}}} - \frac{k_t T_0 \omega_m}{U_{in}{T_{m,\text{all}}}}
$$

$$\mu_m
    = \frac{k_t \omega_m}{U_{in}} - \frac{k_t T_0 \omega_m}{U_{in}{T_{m,\text{all}}}}
$$
